# 00 — Setup & Configuration (Stage 0)
### Pipeline initialization · config v0.3.0 · Colab + Drive or local

**Reads:** `config/project_config.yaml` (v0.3.0). **Writes:** directories, `logs/env_*.json`, startup log. **Next:** `01_test_data_access.ipynb`.

> Config lives on Drive (or workspace root). This notebook never hard-codes scope, dates, or thresholds — it loads them. If the Drive config is missing (first run), the notebook copies from the workspace copy so a fresh runtime self-heals.


In [ ]:
# Cell 1 — Mount Drive + resolve paths (rerunnable; no side effects beyond mount).
import os, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = Path("/content/drive/MyDrive/reddit_embeddings_project")
if IN_COLAB:
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

# Locate project root dynamically:
# 1. Look in current directory and parent directories (if cloned into Colab or Drive)
cwd = Path.cwd().resolve()
found = None
for p in [cwd, *cwd.parents]:
    if (p / "config/project_config.yaml").exists():
        found = p; break

# 2. Check Google Drive root
if found:
    PROJECT_ROOT = found
elif DRIVE_ROOT.exists() and (DRIVE_ROOT / "config/project_config.yaml").exists():
    PROJECT_ROOT = DRIVE_ROOT
else:
    PROJECT_ROOT = DRIVE_ROOT  # first-run destination on Drive

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")

if IN_COLAB and not (PROJECT_ROOT / "src").exists():
    print("[NOTE] If opened directly from GitHub without cloning, clone to Drive once:")
    print("!git clone https://github.com/YOUR_USER/reddit_embeddings_project.git /content/drive/MyDrive/reddit_embeddings_project")


In [ ]:
# Cell 2 — Dependencies: verify, install only what's missing (kept minimal for free Colab).
# Why each: pyyaml=config, requests=API streaming, zstandard=.zst dumps, tqdm=bounded progress.
# gensim is NOT installed here (training notebook only) to keep setup light.
import importlib, subprocess
NEEDED = ["yaml", "requests", "zstandard", "tqdm"]
PIPNAME = {"yaml": "pyyaml", "requests": "requests", "zstandard": "zstandard", "tqdm": "tqdm"}
for mod in NEEDED:
    try:
        importlib.import_module(mod)
        print(f"OK   {mod}")
    except ImportError:
        print(f"MISS {mod} -> pip install {PIPNAME[mod]}")
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", PIPNAME[mod]])
        print(f"INSTALLED {PIPNAME[mod]}")
import yaml, requests, zstandard, tqdm
print("deps ready:", yaml.__version__ if hasattr(yaml, '__version__') else 'yaml-ok')

In [ ]:
# Cell 3 — Helpers: logging (file + concise stdout), atomic write, sha256, config validation.
import csv, hashlib, json, logging, datetime, platform
from pathlib import Path
from src.storage import atomic_write_text, sha256_file

def setup_logger(log_path: Path, level: str = "INFO") -> logging.Logger:
    lg = logging.getLogger("stage0"); lg.setLevel(getattr(logging, level.upper(), logging.INFO))
    lg.handlers.clear()
    fh = logging.FileHandler(log_path); fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
    sh.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    lg.addHandler(fh); lg.addHandler(sh)
    return lg

REQUIRED_KEYS = [["config_version"], ["paths", "drive_root"], ["corpus", "date_range"],
                 ["sources", "priority"], ["slicing", "batch_size"],
                 ["periodization", "atomic_unit"], ["embeddings", "dim"]]

def validate_config(cfg: dict) -> list:
    errs = []
    for ks in REQUIRED_KEYS:
        d = cfg
        for k in ks:
            if not isinstance(d, dict) or k not in d: errs.append("missing: " + ".".join(ks)); break
            d = d[k]
    return errs

print("helpers ready (imported storage utils)")

In [ ]:
# Cell 4 — Create skeleton + load (or first-write) config v0.3.0. RERUN-SAFE: never overwrites existing config.
import shutil
import yaml
ROOT = Path(PROJECT_ROOT)
SUBDIRS = ["config", "manifests", "manifests/pilot", "metadata/monthly_counts",
           "metadata/coverage_reports", "metadata/corpus_statistics", "shards/tokenized",
           "shards/temporary", "shards/quarantine", "models/word2vec", "models/checkpoints",
           "models/fasttext", "vectors", "diagnostics/retrieval", "diagnostics/counts",
           "diagnostics/shards", "diagnostics/model_stability", "diagnostics/semantic_axes",
           "logs", "notebooks"]
for d in SUBDIRS:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

LOG = setup_logger(ROOT / "logs" / f"00_setup__{datetime.datetime.now(datetime.timezone.utc):%Y%m%dT%H%M%SZ}.log",
                   level="INFO")
CFG_PATH = ROOT / "config" / "project_config.yaml"

if not CFG_PATH.exists():
    # If run in Colab and Drive is not yet seeded, copy from repo
    local_cfg = Path("config/project_config.yaml")
    if local_cfg.exists():
        CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_cfg, CFG_PATH)
        print("seeded Drive config from workspace copy")

if CFG_PATH.exists():
    cfg = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
    print(f"loaded existing config {CFG_PATH} version={cfg.get('config_version')}")
else:
    raise FileNotFoundError(f"No config at {CFG_PATH}. Ensure config/project_config.yaml is present.")

errs = validate_config(cfg)
assert not errs, f"config validation failed: {errs}"
CFG_SHA = sha256_file(CFG_PATH)
print(f"config_version={cfg['config_version']}  sha256={CFG_SHA[:16]}...")
print(f"range={cfg['corpus']['date_range']} types={cfg['corpus']['content_types']}")
print(f"periods: atom={cfg['periodization']['atomic_unit']} base={cfg['periodization']['base_unit']} "
      f"min_tokens={cfg['periodization']['min_usable_tokens_per_model']} cap={cfg['reference_sampling']['within_period_cap']['max_train_tokens_per_model']}")

In [ ]:
# Cell 5 — Record environment + inspect existing manifests (resume map). Read-only.
env = {"python": platform.python_version(), "platform": platform.platform(),
       "config_version": cfg["config_version"], "config_sha256": CFG_SHA,
       "utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
try:
    import importlib.metadata as md
    env["packages"] = {p: md.version(p) for p in ["pyyaml", "requests", "zstandard", "tqdm"]}
except Exception as e:
    env["packages_error"] = str(e)
atomic_write_text(ROOT / f"logs/env__{env['utc'][:10]}__cfg-{cfg['config_version']}.json",
                 json.dumps(env, indent=2), None)
print(json.dumps(env, indent=2))
for m in ["manifests/retrieval_manifest.csv", "manifests/pilot/retrieval_manifest.csv",
          "manifests/shard_manifest.csv", "manifests/training_manifest.csv",
          "config/period_definitions.csv"]:
    p = ROOT / m
    if not p.exists():
        print(f"-- {m}: not started"); continue
    with open(p, encoding="utf-8") as f:
        rows = list(csv.DictReader((r for r in f if not r.startswith("#"))))
    from collections import Counter
    print(f"-- {m}: {len(rows)} rows " + str(dict(Counter(r.get('status', '?') for r in rows))))

In [ ]:
# Cell 6 — END-OF-RUN SUMMARY (the only output you need to read).
print("=" * 70)
print(f"SETUP COMPLETE  config={cfg['config_version']} sha={CFG_SHA[:12]} root={ROOT}")
print("completed : skeleton + config validation + env record")
print("remaining : run 01_test_data_access.ipynb (pilot probes + trial shards)")
print("failed    : none (setup asserts loudly instead of partial-writing)")
print("rerun safe: YES — setup never overwrites config/manifests")
print("next      : open 01_test_data_access.ipynb; check pilot subreddits/weeks in its Cell 1")
print("=" * 70)